# 09 - CTD robust reasoning with abstention / no-path SFT

This experiment addresses the failure observed in 08: distractor-aware SFT improved robustness to irrelevant evidence, but both Vanilla and Robust models had **0% no-path accuracy**.

We compare three LoRA-SFT conditions on the same chemical-disjoint evaluation set:

1. **Vanilla** — clean positive paths only.
2. **Distractor-aware** — clean + positive examples containing irrelevant gene-disease edges.
3. **Robust + Abstain** — clean + distractors + explicit no-supported-path examples.

Pilot configuration is intentionally small for Colab/L4: 2,000 training examples per condition, 100 optimization steps, and 100 evaluation examples per condition.

In [ ]:
!pip -q install -U "transformers>=4.55,<5" "datasets>=3.6,<5" "peft>=0.17,<1" "trl==0.29.1" "accelerate>=1.10,<2" "bitsandbytes>=0.46,<1" "torchao>=0.16,<1"


In [ ]:
import os, re, random
import pandas as pd
import torch
from datasets import Dataset
from google.colab import files

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab.')

CHEM_GENE='/content/CTD_chem_gene_ixns.tsv.gz'
GENE_DISEASE='/content/CTD_curated_genes_diseases.tsv.gz'
if not os.path.exists(CHEM_GENE) or not os.path.exists(GENE_DISEASE):
    print('Upload CTD_chem_gene_ixns.tsv.gz and CTD_curated_genes_diseases.tsv.gz')
    files.upload()
assert os.path.exists(CHEM_GENE) and os.path.exists(GENE_DISEASE)
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
chem_cols=['ChemicalName','ChemicalID','CasRN','GeneSymbol','GeneID','GeneForms','Organism','OrganismID','Interaction','InteractionActions','PubMedIDs']
gd_cols=['GeneSymbol','GeneID','DiseaseName','DiseaseID','DirectEvidence','InferenceChemicalName','InferenceChemicalID','OmimIDs','PubMedIDs']
chem=pd.read_csv(CHEM_GENE,sep='\t',comment='#',header=None,names=chem_cols,dtype=str,low_memory=False)
gd=pd.read_csv(GENE_DISEASE,sep='\t',comment='#',header=None,names=gd_cols,dtype=str,low_memory=False)
chem=chem[chem['OrganismID'].fillna('').str.strip().eq('9606')].copy()
chem=chem.dropna(subset=['ChemicalName','ChemicalID','GeneSymbol','GeneID'])
gd=gd.dropna(subset=['GeneID','DiseaseName','DiseaseID']).drop_duplicates(['GeneID','DiseaseID'])
chem['GeneID']=chem['GeneID'].str.replace(r'\.0$','',regex=True); gd['GeneID']=gd['GeneID'].str.replace(r'\.0$','',regex=True)
pairs=chem.merge(gd[['GeneID','DiseaseName','DiseaseID']],on='GeneID',how='inner')
pairs=pairs[['ChemicalName','ChemicalID','GeneSymbol','GeneID','DiseaseName','DiseaseID']].dropna().drop_duplicates(['ChemicalID','GeneID','DiseaseID']).reset_index(drop=True)
print('2-hop paths:',len(pairs))


In [ ]:
# Chemical-disjoint split and compact pilot pools.
rng=random.Random(42)
chems=pairs['ChemicalID'].drop_duplicates().tolist(); rng.shuffle(chems)
eval_chems=set(chems[:max(1,int(.1*len(chems)))])
train_pool=pairs[~pairs['ChemicalID'].isin(eval_chems)].sample(frac=1,random_state=42).head(3000).reset_index(drop=True)
eval_pool=pairs[pairs['ChemicalID'].isin(eval_chems)].sample(frac=1,random_state=43).head(100).reset_index(drop=True)
print('Train pool:',len(train_pool),'Eval pool:',len(eval_pool))

# Cache gene-disease edges once; avoids repeated dataframe filtering.
edge_pool=list({(str(g),str(d)) for g,d in pairs[['GeneSymbol','DiseaseName']].itertuples(index=False,name=None)})
print('Cached gene-disease edges:',len(edge_pool))


In [ ]:
def clean_prompt(row):
    return (f'Evidence 1: {row.ChemicalName} has a CTD chemical-gene relationship with gene {row.GeneSymbol}.\n'
            f'Evidence 2: gene {row.GeneSymbol} is linked to disease {row.DiseaseName}.\n'
            f'Question: Is there a supported disease path from {row.ChemicalName} through gene {row.GeneSymbol}? '
            'If supported, return `Disease: <name>` and `Path: Chemical -> Gene -> Disease`. Otherwise return `No supported path.`')

def positive_answer(row):
    return f'Disease: {row.DiseaseName}. Path: {row.ChemicalName} -> {row.GeneSymbol} -> {row.DiseaseName}.'

def valid_distractors(row,k,rng):
    # Random rejection sampling from cached edges is much faster than filtering the full dataframe per example.
    out=[]; seen=set(); attempts=0
    while len(out)<k and attempts<500:
        g,d=rng.choice(edge_pool); attempts+=1
        if g==row.GeneSymbol or d==row.DiseaseName or (g,d) in seen: continue
        seen.add((g,d)); out.append((g,d))
    return out if len(out)==k else None

def distractor_prompt(row,k,rng):
    ds=valid_distractors(row,k,rng)
    if ds is None: return None
    edges=[f'{row.GeneSymbol} -> {row.DiseaseName}']+[f'{g} -> {d}' for g,d in ds]; rng.shuffle(edges)
    return (f'Evidence A: {row.ChemicalName} has a CTD chemical-gene relationship with gene {row.GeneSymbol}.\n'
            +'Gene-disease evidence:\n- '+'\n- '.join(edges)
            +f'\nQuestion: Is there a supported disease path from {row.ChemicalName} through gene {row.GeneSymbol}? '
             'If supported, return `Disease: <name>` and a path. Otherwise return `No supported path.`')

def no_path_prompt(row,k,rng):
    ds=valid_distractors(row,k,rng)
    if ds is None: return None
    edges=[f'{g} -> {d}' for g,d in ds]; rng.shuffle(edges)
    return (f'Evidence A: {row.ChemicalName} has a CTD chemical-gene relationship with gene {row.GeneSymbol}.\n'
            +'Gene-disease evidence:\n- '+'\n- '.join(edges)
            +f'\nQuestion: Is there a supported disease path from {row.ChemicalName} through gene {row.GeneSymbol}? '
             'If no supplied gene-disease edge uses the queried gene, return exactly `No supported path.`')

NO_PATH_ANSWER='No supported path.'


In [ ]:
from transformers import AutoModelForCausalLM,AutoTokenizer
MODEL_NAME='Qwen/Qwen2.5-0.5B-Instruct'
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='left'
dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

def render(prompt,answer=None):
    msgs=[{'role':'user','content':prompt}]
    if answer is not None: msgs.append({'role':'assistant','content':answer})
    return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=answer is None)


In [ ]:
# Build three balanced training conditions.
MAX_TRAIN=2000
def make_training(condition,seed):
    rr=random.Random(seed); rows=[]
    source=train_pool.sample(n=min(MAX_TRAIN,len(train_pool)),random_state=seed).reset_index(drop=True)
    for row in source.itertuples(index=False):
        u=rr.random()
        if condition=='vanilla':
            p,a=clean_prompt(row),positive_answer(row)
        elif condition=='distractor':
            if u<.5:
                p=distractor_prompt(row,3,rr); a=positive_answer(row)
                if p is None: p=clean_prompt(row)
            else: p,a=clean_prompt(row),positive_answer(row)
        elif condition=='abstain':
            # 40% clean, 30% positive distractor, 30% no-path.
            if u<.4: p,a=clean_prompt(row),positive_answer(row)
            elif u<.7:
                p=distractor_prompt(row,3,rr); a=positive_answer(row)
                if p is None: p=clean_prompt(row)
            else:
                p=no_path_prompt(row,3,rr); a=NO_PATH_ANSWER
                if p is None: p=clean_prompt(row); a=positive_answer(row)
        rows.append({'text':render(p,a)})
    return Dataset.from_list(rows)

vanilla_ds=make_training('vanilla',1)
distractor_ds=make_training('distractor',2)
abstain_ds=make_training('abstain',3)
print(len(vanilla_ds),len(distractor_ds),len(abstain_ds))


In [ ]:
# Fixed evaluation sets. Positive distractor and no-path cases use the same queried chemicals/genes.
eval_sets={'clean':[],'distractor_1':[],'distractor_3':[],'distractor_5':[],'distractor_10':[],'no_path_1':[],'no_path_3':[],'no_path_5':[],'no_path_10':[]}
rr=random.Random(999)
for row in eval_pool.itertuples(index=False):
    base={'target_disease':row.DiseaseName,'target_gene':row.GeneSymbol,'target_chemical':row.ChemicalName}
    eval_sets['clean'].append({**base,'prompt':clean_prompt(row),'kind':'positive'})
    for k in [1,3,5,10]:
        p=distractor_prompt(row,k,rr)
        if p: eval_sets[f'distractor_{k}'].append({**base,'prompt':p,'kind':'positive'})
        np=no_path_prompt(row,k,rr)
        if np: eval_sets[f'no_path_{k}'].append({**base,'prompt':np,'kind':'no_path'})
for name,x in eval_sets.items(): print(name,len(x))


In [ ]:
from peft import LoraConfig
from trl import SFTConfig,SFTTrainer
lora=LoraConfig(r=8,lora_alpha=16,lora_dropout=.05,target_modules=['q_proj','k_proj','v_proj','o_proj'],bias='none',task_type='CAUSAL_LM')

def train_adapter(ds,outdir):
    m=AutoModelForCausalLM.from_pretrained(MODEL_NAME,dtype=dtype).cuda(); m.config.use_cache=False
    args=SFTConfig(output_dir=outdir,per_device_train_batch_size=4,gradient_accumulation_steps=2,max_steps=100,learning_rate=2e-4,logging_steps=20,save_strategy='no',report_to='none',packing=False,gradient_checkpointing=False,fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported())
    trainer=SFTTrainer(model=m,args=args,train_dataset=ds,processing_class=tokenizer,peft_config=lora)
    trainer.train(); trainer.save_model(outdir); tokenizer.save_pretrained(outdir)
    del trainer,m; torch.cuda.empty_cache()

for label,ds,path in [('Vanilla',vanilla_ds,'./outputs/09-vanilla'),('Distractor',distractor_ds,'./outputs/09-distractor'),('Robust+Abstain',abstain_ds,'./outputs/09-abstain')]:
    print('Training',label); train_adapter(ds,path)


In [ ]:
def generate_batched(model,prompts,batch_size=16,max_new_tokens=56):
    outs=[]; model.eval()
    for s in range(0,len(prompts),batch_size):
        texts=[render(p) for p in prompts[s:s+batch_size]]
        enc=tokenizer(texts,return_tensors='pt',padding=True,truncation=True,max_length=384); enc={k:v.to(model.device) for k,v in enc.items()}
        with torch.inference_mode(): out=model.generate(**enc,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.pad_token_id)
        n=enc['input_ids'].shape[1]; outs.extend(tokenizer.batch_decode(out[:,n:],skip_special_tokens=True))
    return outs

def normalize(x): return re.sub(r'[^a-z0-9]+',' ',str(x).lower()).strip()
def score(items,preds):
    hits=[]
    for item,p in zip(items,preds):
        q=normalize(p)
        if item['kind']=='no_path': hits.append('no supported path' in q)
        else: hits.append(normalize(item['target_disease']) in q)
    return sum(hits)/len(hits) if hits else float('nan')


In [ ]:
from peft import PeftModel
def evaluate_adapter(path,label):
    base=AutoModelForCausalLM.from_pretrained(MODEL_NAME,dtype=dtype).cuda()
    m=PeftModel.from_pretrained(base,path); m.eval(); result={}
    for name,items in eval_sets.items():
        preds=generate_batched(m,[x['prompt'] for x in items])
        result[name]=score(items,preds); print(f'{label:16s} {name:16s} {result[name]:.3f}')
    del m,base; torch.cuda.empty_cache(); return result

vanilla_results=evaluate_adapter('./outputs/09-vanilla','Vanilla')
distractor_results=evaluate_adapter('./outputs/09-distractor','Distractor')
abstain_results=evaluate_adapter('./outputs/09-abstain','Robust+Abstain')


In [ ]:
print('\n09 ROBUSTNESS + ABSTENTION COMPARISON')
print('='*92)
print(f"{'Condition':<18}{'Vanilla':>12}{'Distractor':>14}{'Robust+Abstain':>18}")
print('-'*92)
for name in eval_sets:
    print(f"{name:<18}{vanilla_results[name]:>12.3f}{distractor_results[name]:>14.3f}{abstain_results[name]:>18.3f}")


## Interpretation

The key question is whether explicit no-path training fixes abstention without sacrificing positive-path robustness. The desired pattern is: Vanilla performs well on clean examples but poorly under distractors/no-path; Distractor-aware SFT improves positive-path robustness but still fails to abstain; Robust+Abstain preserves clean/distractor performance while substantially increasing no-path accuracy.

This is still a pilot: the next paper-quality stage should use multiple random seeds, larger held-out sets, stricter entity-disjoint splits, and confidence intervals/statistical tests.